# Priorización de gestión — caída de consumo

Tercer notebook de la fase. No detecta nada nuevo: toma la salida del estudio
de caída (`estudio_caida_consumo.parquet`) y la convierte en **listas de
trabajo repartibles**, cruzándola con los datos comerciales del cliente.

Es liviano: lee resultados ya calculados, no reprocesa la serie ni reagrupa.

## Qué agrega

**Ciclo (0–99).** El ciclo es la ruta de lectura, así que es el eje que
convierte 37.600 clientes dispersos en órdenes de trabajo por cuadrilla:
"estos 180 están todos en el ciclo 12, va un equipo".

**Pesos, con la tarifa real de cada cliente.** El histórico trae
`tarifa_aplicada_kwh`, que viene de la columna original *"Tarifa Aplicada
($/kWh)"* agregada como mediana por cliente y periodo. Es la tarifa que se le
aplica a ese cliente, no un promedio del sector.

Se usa el último valor **real** de cada cliente —no el último registro, que
podría traer la tarifa vacía— y **no se imputa nada**: quien no tenga tarifa
propia queda *sin valorar* y se reporta aparte, con sus kWh a la vista.
Rellenarle el promedio de su clase produciría una cifra en pesos que no
corresponde a ningún cliente real, y esas cifras terminan en una presentación
como si lo fueran.

El campo se **audita antes de usarse**: si no se comporta como un valor por
kWh, el notebook lo dice y prioriza por kWh en vez de reportar pesos.

**Cortes de gestión.** Clase de servicio (RS/CR/OF/ID…), estrato, zona
rural/urbana y tramo de consumo. Cada uno reparte el trabajo a un área
distinta.

## Dos vistas

- **Operativa** — ordenada por ciclo, y dentro de cada ciclo por valor en
  riesgo. Para la cuadrilla.
- **Gerencial** — ranking global por valor en riesgo, sin importar el ciclo.
  Para decidir dónde poner la atención primero.

## Salida

- `gestion_caida_operativa.csv` — la lista por ciclo, para repartir.
- `gestion_caida_gerencial.csv` — el ranking global por valor.
- `resumen_gestion_por_ciclo.csv` — cuánto vale cada ruta.
- `resumen_gestion_por_corte.csv` — clase, estrato, zona y tramo.


In [ ]:
# ============================================================
# 1. LIBRERÍAS Y RUTAS
# ============================================================

from pathlib import Path
import os
import re
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from IPython.display import display

# La ruta de datos se puede fijar desde fuera (pipeline_mensual.py, app web)
# con la variable de entorno EBSA_DATOS. Si no existe, se usa la de siempre.
BASE_DIR = Path(os.environ.get("EBSA_DATOS", r"C:\Users\Home\Documents\Datos_Ebsa"))

PREPROC_DIR = BASE_DIR / "03_serie_modelado"
AGRUP_DIR = BASE_DIR / "05_segmentos_clientes"
CAIDA_DIR = BASE_DIR / "06_estudio_caida"
PROCESADO_DIR = BASE_DIR / "01_historico_procesado"

GESTION_DIR = BASE_DIR / "07_gestion_caida"
GESTION_DIR.mkdir(parents=True, exist_ok=True)

RUTA_CAIDA = CAIDA_DIR / "estudio_caida_consumo.parquet"
RUTA_CLUSTERS = AGRUP_DIR / "clientes_clusters_consumo.parquet"
RUTA_NIU_CICLO = PREPROC_DIR / "niu_ciclo.parquet"

RUTA_OPERATIVA = GESTION_DIR / "gestion_caida_operativa.csv"
RUTA_GERENCIAL = GESTION_DIR / "gestion_caida_gerencial.csv"
RUTA_RESUMEN_CICLO = GESTION_DIR / "resumen_gestion_por_ciclo.csv"
RUTA_RESUMEN_CORTE = GESTION_DIR / "resumen_gestion_por_corte.csv"
RUTA_AUDITORIA_TARIFA = GESTION_DIR / "auditoria_tarifa.csv"
RUTA_DIAGNOSTICO_SESGO = GESTION_DIR / "diagnostico_sesgo_pronostico.csv"

# Historial: una copia de la lista por cada corte. La lista "actual" se
# sobreescribe cada mes; estas no. Con ellas se sabe si un cliente es nuevo
# en la lista o lleva meses apareciendo, y se puede evaluar después qué pasó
# con cada uno (Evaluacion_retroalimentacion_gestion.ipynb).
HISTORIAL_GESTION_DIR = GESTION_DIR / "historial"
HISTORIAL_GESTION_DIR.mkdir(parents=True, exist_ok=True)
RUTA_ENTRADAS_SALIDAS = GESTION_DIR / "resumen_entradas_salidas_lista.csv"

# --- Pronóstico del modelo de predicción ---
# Carpeta donde el notebook de backtest/reentrenamiento guarda el pronóstico
SALIDA_MODELO_DIR = (
    BASE_DIR / "04_pronostico" / "modelo_final"
)
RUTA_PRONOSTICO = SALIDA_MODELO_DIR / "predicciones_segmentadas_optimizadas_6_meses.parquet"
RUTA_METRICAS_PERFIL = SALIDA_MODELO_DIR / "metricas_sistema_por_perfil_horizonte_optimizado.csv"

# El pronóstico se usa TAL COMO SALE del modelo de producción, con su corte en
# el último mes que entrega la empresa. No se recalcula: ese es el dato con el
# que el modelo va a operar de verdad cada mes.
USAR_PRONOSTICO = True

# ¿Se evalúa trayectoria en rurales? Lo decide la BRECHA MEDIDA entre el
# sesgo del pronóstico en rurales estables y en urbanos estables (celda 6c).
# Si la brecha supera este tope, el pronóstico rural arranca de una base
# distinta a la urbana (típicamente el borde provisional) y no es comparable.
#   None  -> automático según la brecha (recomendado)
#   True  -> forzar exclusión de rurales
#   False -> forzar inclusión
EXCLUIR_RURAL_DE_TRAYECTORIA = None
BRECHA_MAXIMA_ACEPTABLE_PP = 5.0

# Alcance: qué veredictos entran a la lista de gestión
VEREDICTOS_GESTION = ["CAIDA_CONFIRMADA", "CAIDA_SOSTENIDA"]

# Rango plausible para el valor del kWh en Colombia (COP). Solo se usa para
# AUDITAR el campo, no para corregirlo: si la mediana cae fuera, el notebook
# avisa y no calcula pesos, en vez de producir cifras sin sentido.
TARIFA_COP_MIN_PLAUSIBLE = 100.0
TARIFA_COP_MAX_PLAUSIBLE = 3000.0

print("Estudio de caída :", RUTA_CAIDA)
print("Segmentos        :", RUTA_CLUSTERS)
print("Ciclo por NIU    :", RUTA_NIU_CICLO)
print("Históricos       :", PROCESADO_DIR)
print("Salidas          :", GESTION_DIR)
print()
print("Alcance de la lista:", ", ".join(VEREDICTOS_GESTION))


## Cargar resultados ya calculados

In [ ]:
# ============================================================
# 2. CARGAR ESTUDIO DE CAÍDA Y SEGMENTOS
# ============================================================

for ruta in (RUTA_CAIDA, RUTA_CLUSTERS, RUTA_NIU_CICLO):
    if not ruta.exists():
        raise FileNotFoundError(
            f"No existe:\n{ruta}\n"
            "Corre primero el notebook que lo produce."
        )

caida = pd.read_parquet(RUTA_CAIDA, engine="pyarrow")
caida["NIU"] = caida["NIU"].astype("string").str.strip()

if "fecha_corte" not in caida.columns:
    raise ValueError(
        "El estudio de caída no trae la columna fecha_corte. Vuelve a correr "
        "Estudio_caida_consumo.ipynb con la versión actual antes de priorizar."
    )
FECHA_CORTE = pd.Timestamp(caida["fecha_corte"].iloc[0]).to_period("M").to_timestamp()
ETIQUETA_CORTE = f"{FECHA_CORTE:%Y-%m}"
print(f"Corte de la lista (fin de ventana del estudio de caída): {ETIQUETA_CORTE}")

clusters = pd.read_parquet(
    RUTA_CLUSTERS,
    columns=["NIU", "tramo_consumo", "segmento_prioridad_texto"],
    engine="pyarrow",
)
clusters["NIU"] = clusters["NIU"].astype("string").str.strip()

caida = caida.merge(clusters, on="NIU", how="left")

print("ESTUDIO DE CAÍDA CARGADO")
print("-" * 60)
print(f"Clientes: {len(caida):,}")
display(caida["veredicto"].value_counts().rename("n_clientes").to_frame())

gestion = caida[caida["veredicto"].isin(VEREDICTOS_GESTION)].copy()

print(f"\nEntran a la lista de gestión: {len(gestion):,} clientes")
print(f"kWh/mes en riesgo           : {gestion['perdida_kwh_mes'].sum():,.0f}")


## Ciclo de lectura (0–99)

In [ ]:
# ============================================================
# 3. CICLO POR NIU
# ============================================================
# El ciclo es la ruta de lectura. Es el eje que hace repartible la lista.
# ============================================================

ciclo_por_niu = pd.read_parquet(RUTA_NIU_CICLO, engine="pyarrow")
ciclo_por_niu["NIU"] = ciclo_por_niu["NIU"].astype("string").str.strip()

columnas_ciclo = [c for c in ["NIU", "ciclo", "es_rural"] if c in ciclo_por_niu.columns]
if "ciclo" not in columnas_ciclo:
    raise ValueError(
        f"niu_ciclo.parquet no tiene columna 'ciclo'. Tiene: "
        f"{list(ciclo_por_niu.columns)}"
    )

ciclo_por_niu = ciclo_por_niu[columnas_ciclo].drop_duplicates("NIU")

gestion = gestion.merge(ciclo_por_niu, on="NIU", how="left")

n_sin_ciclo = int(gestion["ciclo"].isna().sum())

print("COBERTURA DE CICLO")
print("-" * 60)
print(f"Con ciclo identificado : {len(gestion) - n_sin_ciclo:,}")
print(f"Sin ciclo              : {n_sin_ciclo:,}")

if n_sin_ciclo:
    print(
        "\nLos que no tienen ciclo van a un grupo 'SIN_CICLO' al final de la\n"
        "lista operativa: no se pueden asignar a una ruta, hay que resolverlos\n"
        "primero contra el sistema comercial."
    )

gestion["ciclo"] = gestion["ciclo"].astype("Float64")

# Validar el rango 0-99 que define el negocio
fuera_de_rango = gestion["ciclo"].dropna()
fuera_de_rango = fuera_de_rango[(fuera_de_rango < 0) | (fuera_de_rango > 99)]
if len(fuera_de_rango):
    print(f"\n⚠ {len(fuera_de_rango):,} clientes con ciclo fuera del rango 0-99.")

print(f"\nCiclos distintos presentes: {int(gestion['ciclo'].nunique())}")
display(
    gestion["ciclo"].value_counts().head(15)
    .rename("n_clientes").to_frame()
    .rename_axis("ciclo")
)


## Datos comerciales: tarifa, estrato y clase de servicio

`tarifa_aplicada_kwh` no lo hemos usado nunca. Antes de multiplicar kWh por
ese número y presentar pesos, se audita: cobertura de nulos, distribución, si
cae dentro de un rango plausible para el valor del kWh, y si varía en el
tiempo. Si no pasa la auditoría, el notebook sigue trabajando en kWh y lo
dice, en vez de producir cifras en pesos que nadie podría defender.


In [ ]:
# ============================================================
# 4. LEER TARIFA, ESTRATO Y CLASE DE SERVICIO DEL HISTÓRICO
# ============================================================

patron = re.compile(r"^historico_(20\d{2})\.parquet$", re.IGNORECASE)
archivos = sorted(
    [p for p in PROCESADO_DIR.glob("historico_*.parquet") if patron.match(p.name)],
    key=lambda p: int(patron.match(p.name).group(1)),
)
if not archivos:
    raise FileNotFoundError(f"No hay historico_YYYY.parquet en:\n{PROCESADO_DIR}")

COLS_COMERCIAL = ["NIU", "periodo", "tarifa_aplicada_kwh", "estrato", "clase_servicio"]

partes = []
for ruta in archivos:
    disponibles = pq.ParquetFile(ruta).schema_arrow.names
    faltan = [c for c in COLS_COMERCIAL if c not in disponibles]
    if faltan:
        print(f"⚠ {ruta.name} no tiene {faltan}, se omite.")
        continue
    partes.append(pd.read_parquet(ruta, columns=COLS_COMERCIAL, engine="pyarrow"))

if not partes:
    raise ValueError(
        "Ningún histórico tiene los campos comerciales. Revisa los nombres "
        f"exactos; se buscaban: {COLS_COMERCIAL}"
    )

comercial = pd.concat(partes, ignore_index=True)
comercial["NIU"] = comercial["NIU"].astype("string").str.strip()
comercial["periodo"] = pd.to_datetime(comercial["periodo"], errors="coerce")
comercial["tarifa_aplicada_kwh"] = pd.to_numeric(
    comercial["tarifa_aplicada_kwh"], errors="coerce"
)

print("DATOS COMERCIALES CARGADOS")
print("-" * 60)
print(f"Filas      : {len(comercial):,}")
print(f"NIU únicos : {comercial['NIU'].nunique():,}")

del partes
gc.collect()


In [ ]:
# ============================================================
# 5. AUDITAR tarifa_aplicada_kwh ANTES DE CALCULAR PESOS
# ============================================================

tarifa = comercial["tarifa_aplicada_kwh"]
n_filas = len(tarifa)
n_nulos = int(tarifa.isna().sum())
n_cero = int((tarifa == 0).sum())

no_nulos = tarifa.dropna()
no_nulos = no_nulos[no_nulos > 0]

mediana_tarifa = float(no_nulos.median()) if len(no_nulos) else np.nan

print("COBERTURA")
print("-" * 70)
print(f"Filas          : {n_filas:,}")
print(f"Nulos          : {n_nulos:,} ({n_nulos / n_filas * 100:.2f}%)")
print(f"Ceros          : {n_cero:,} ({n_cero / n_filas * 100:.2f}%)")
print(f"Utilizables    : {len(no_nulos):,}")

print("\nDISTRIBUCIÓN DEL VALOR")
print("-" * 70)
display(no_nulos.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).to_frame().T)

# ¿Se comporta como un valor por kWh?
en_rango = (
    TARIFA_COP_MIN_PLAUSIBLE <= mediana_tarifa <= TARIFA_COP_MAX_PLAUSIBLE
    if np.isfinite(mediana_tarifa) else False
)

print("\n¿VARÍA EN EL TIEMPO?")
print("-" * 70)
por_anio = (
    comercial.assign(anio=comercial["periodo"].dt.year)
    .loc[comercial["tarifa_aplicada_kwh"] > 0]
    .groupby("anio")["tarifa_aplicada_kwh"]
    .median()
    .round(2)
)
display(por_anio.rename("mediana_tarifa").to_frame())

print("\nVEREDICTO DE LA AUDITORÍA")
print("=" * 70)
print(f"Mediana del campo: {mediana_tarifa:,.2f}")
print(f"Rango plausible para $/kWh: "
      f"{TARIFA_COP_MIN_PLAUSIBLE:,.0f} - {TARIFA_COP_MAX_PLAUSIBLE:,.0f}")

USAR_PESOS = bool(en_rango)

if USAR_PESOS:
    print("\n✓ El campo se comporta como un valor por kWh. Se calcularán pesos.")
    if por_anio.is_monotonic_increasing:
        print("  Además crece año a año, consistente con indexación tarifaria.")
else:
    print(
        "\n⚠ La mediana cae FUERA del rango plausible para un valor por kWh.\n"
        "  El campo puede estar en otra unidad, o ser un valor facturado total.\n"
        "  La priorización seguirá en kWh y NO se reportarán pesos."
    )

auditoria_tarifa = pd.DataFrame([
    ["filas", n_filas],
    ["nulos", n_nulos],
    ["ceros", n_cero],
    ["mediana", round(mediana_tarifa, 2) if np.isfinite(mediana_tarifa) else None],
    ["p01", round(float(no_nulos.quantile(0.01)), 2) if len(no_nulos) else None],
    ["p99", round(float(no_nulos.quantile(0.99)), 2) if len(no_nulos) else None],
    ["usar_pesos", int(USAR_PESOS)],
], columns=["metrica", "valor"])

auditoria_tarifa.to_csv(RUTA_AUDITORIA_TARIFA, index=False, encoding="utf-8-sig")
print("\nGuardado:", RUTA_AUDITORIA_TARIFA)


In [ ]:
# ============================================================
# 6. TARIFA, ESTRATO Y CLASE VIGENTES POR CLIENTE
# ============================================================
# Se toma el registro más reciente de cada NIU: es la condición comercial
# vigente, que es la que aplica para valorar una pérdida de hoy.
# ============================================================

# La tarifa vigente es el ÚLTIMO valor REAL del cliente, no el último
# registro: si el registro más reciente trae la tarifa vacía, tomarlo
# dejaría sin valorar a un cliente que sí tiene tarifa unos meses atrás.
tarifa_vigente = (
    comercial.loc[comercial["tarifa_aplicada_kwh"] > 0, ["NIU", "periodo", "tarifa_aplicada_kwh"]]
    .sort_values("periodo")
    .drop_duplicates("NIU", keep="last")
    .set_index("NIU")
)

# Estrato y clase sí se toman del registro más reciente disponible
atributos_vigentes = (
    comercial.sort_values("periodo")
    .drop_duplicates("NIU", keep="last")
    .set_index("NIU")[["estrato", "clase_servicio"]]
)

gestion = gestion.merge(
    atributos_vigentes, left_on="NIU", right_index=True, how="left",
)
gestion = gestion.merge(
    tarifa_vigente[["tarifa_aplicada_kwh"]].rename(
        columns={"tarifa_aplicada_kwh": "tarifa_kwh"}
    ),
    left_on="NIU", right_index=True, how="left",
)
gestion["periodo_tarifa"] = gestion["NIU"].map(tarifa_vigente["periodo"])

gestion["clase_servicio"] = (
    gestion["clase_servicio"].astype("string").str.strip().fillna("SIN_DATO")
)
gestion["estrato"] = pd.to_numeric(gestion["estrato"], errors="coerce").astype("Int64")

print("CLASE DE SERVICIO")
print("-" * 60)
display(gestion["clase_servicio"].value_counts().rename("n_clientes").to_frame())

print("\nESTRATO")
print("-" * 60)
display(
    gestion["estrato"].value_counts(dropna=False).sort_index()
    .rename("n_clientes").to_frame()
)
print("(estrato 0 corresponde a los no residenciales, no es dato faltante)")

# ============================================================
# VALOR EN RIESGO — SOLO CON LA TARIFA REAL DEL CLIENTE
# ============================================================
# No se imputa ninguna tarifa. Un cliente sin tarifa propia queda SIN VALORAR
# y se reporta aparte: inventarle el promedio de su clase produciría una cifra
# en pesos que no corresponde a nadie, y esas cifras terminan en una
# presentación como si fueran reales.
# ============================================================

gestion["tiene_tarifa"] = gestion["tarifa_kwh"].notna() & (gestion["tarifa_kwh"] > 0)

if USAR_PESOS:
    gestion["valor_riesgo_mes"] = np.where(
        gestion["tiene_tarifa"],
        (gestion["perdida_kwh_mes"] * gestion["tarifa_kwh"]).round(0),
        np.nan,
    )
else:
    gestion["valor_riesgo_mes"] = np.nan

n_con = int(gestion["tiene_tarifa"].sum())
n_sin = len(gestion) - n_con

print("\nCOBERTURA DE TARIFA EN LA LISTA DE GESTIÓN")
print("-" * 70)
print(f"Con tarifa real  : {n_con:,} ({n_con / len(gestion) * 100:.1f}%)")
print(f"Sin tarifa       : {n_sin:,} ({n_sin / len(gestion) * 100:.1f}%)")

if n_sin:
    kwh_sin = gestion.loc[~gestion["tiene_tarifa"], "perdida_kwh_mes"].sum()
    print(
        f"\nEsos {n_sin:,} clientes suman {kwh_sin:,.0f} kWh/mes que quedan SIN VALORAR.\n"
        "No se les imputa tarifa: aparecen en la lista con el valor vacío y hay\n"
        "que resolver su tarifa contra el sistema comercial para poder valorarlos."
    )

if USAR_PESOS and n_con:
    valorado = gestion["valor_riesgo_mes"].sum()
    print(f"\nVALOR EN RIESGO (solo clientes con tarifa real)")
    print("-" * 70)
    print(f"  ${valorado:,.0f} / mes sobre {n_con:,} clientes")
    print("\nTarifa aplicada — distribución en la lista:")
    display(
        gestion.loc[gestion["tiene_tarifa"], "tarifa_kwh"]
        .describe(percentiles=[0.05, 0.5, 0.95]).round(2).to_frame().T
    )
    antiguedad = (
        pd.Timestamp(gestion["periodo_tarifa"].max()) - gestion["periodo_tarifa"]
    ).dt.days
    print(f"Antigüedad de la tarifa usada: mediana {antiguedad.median():.0f} días, "
          f"máximo {antiguedad.max():.0f} días")

# ============================================================
# CLAVE DE ORDEN — distinta del valor reportado
# ============================================================
# valor_riesgo_mes se REPORTA y queda vacío sin tarifa real.
# Pero ordenar por él hundiría al fondo de su ruta a un cliente que perdió
# miles de kWh, solo porque le falta un dato administrativo. Para ORDENAR se
# usa el valor real cuando existe, y kWh x la mediana de tarifa cuando no.
# Es una clave de posición, nunca una cifra que se reporte ni se sume.
# ============================================================

if USAR_PESOS:
    gestion["valor_orden"] = np.where(
        gestion["tiene_tarifa"],
        gestion["valor_riesgo_mes"],
        gestion["perdida_kwh_mes"] * mediana_tarifa,
    )
    COLUMNA_ORDEN = "valor_orden"
    print(
        f"\nOrden: por valor real; los {n_sin:,} sin tarifa se posicionan con "
        f"kWh x {mediana_tarifa:,.0f}\n(solo para no perderlos de vista en la "
        "lista; su valor reportado sigue vacío)."
    )
else:
    gestion["valor_orden"] = gestion["perdida_kwh_mes"]
    COLUMNA_ORDEN = "perdida_kwh_mes"
print(f"\nSe priorizará por: {COLUMNA_ORDEN}")


## Trayectoria: cruzar con el pronóstico del modelo

El estudio de caída mira hacia atrás. El modelo de predicción mira hacia
adelante. Juntarlos separa dos situaciones que exigen cosas distintas: el
cliente que cayó y no da señales de recuperarse, y el que cayó pero el modelo
proyecta que vuelve.

### Dos límites que hay que respetar

**Solo el mes siguiente.** La trayectoria compara el consumo reciente con el
pronóstico a **un mes**, no con promedios a 3 o 6. Es el horizonte más preciso
del modelo (P1: 15% de error a 1 mes contra 26% a 3 y 40% a 6), es el que se
gestiona en la práctica, y su mes de examen queda lejos del borde provisional.

**El error del modelo.** Una caída proyectada menor que el WAPE del perfil del
cliente está dentro del ruido y no significa nada. Por eso la trayectoria solo
se afirma cuando el cambio proyectado supera ese WAPE; el resto queda
explícitamente sin señal, en vez de fingir una conclusión.

**El borde truncado en rurales.** El pronóstico arranca desde el último mes
que entrega la empresa, y en ese mes los rurales aparecen con la mitad de su
consumo real porque la lectura trimestral que lo cerraría todavía no ocurrió.
El modelo aprendió con meses completos y al predecir recibe uno incompleto,
así que para rurales proyecta desde una base artificialmente baja.

No se corrige el pronóstico —ese es el dato con el que el modelo va a operar
en producción todos los meses— pero **sí se mide el sesgo** y los rurales
quedan fuera de la clasificación de trayectoria, marcados como tales. Usarlos
diría "cayó y va a seguir cayendo" cuando ambas mitades vienen del mismo
borde incompleto.


In [ ]:
# ============================================================
# 6b. CARGAR PRONÓSTICO Y ERROR DEL MODELO
# ============================================================

HAY_PRONOSTICO = USAR_PRONOSTICO and RUTA_PRONOSTICO.exists()

if not HAY_PRONOSTICO:
    if USAR_PRONOSTICO:
        raise FileNotFoundError(
            f"No se encontró el pronóstico:\n{RUTA_PRONOSTICO}\n"
            "Corre primero Backtest_y_reentrenamiento_final_optimizado.ipynb, "
            "o pon USAR_PRONOSTICO = False para priorizar sin trayectoria."
        )
    print("USAR_PRONOSTICO = False: se prioriza sin trayectoria.")
else:
    pronostico = pd.read_parquet(
        RUTA_PRONOSTICO,
        columns=["NIU", "fecha_corte", "perfil", "fecha_pred_1m", "pred_1m_kwh"],
        engine="pyarrow",
    )
    pronostico["NIU"] = pronostico["NIU"].astype("string").str.strip()

    metricas_perfil = pd.read_csv(RUTA_METRICAS_PERFIL)
    wape_por_grupo = (
        metricas_perfil.set_index(["perfil", "horizonte"])["WAPE_final_pct"]
    )

    corte_pronostico = pd.to_datetime(pronostico["fecha_corte"]).max()

    print("PRONÓSTICO CARGADO")
    print("-" * 70)
    print(f"Clientes con pronóstico : {pronostico['NIU'].nunique():,}")
    print(f"Corte del pronóstico    : {corte_pronostico:%Y-%m}")
    print("\nWAPE del backtest por perfil, horizonte 1 (el que usa la trayectoria):")
    display(
        metricas_perfil[metricas_perfil["horizonte"].eq(1)]
        [["perfil", "horizonte", "WAPE_final_pct", "sesgo_final_pct"]]
        .round(2).sort_values(["horizonte", "perfil"])
    )


In [ ]:
# ============================================================
# 6c. MEDIR EL SESGO DEL PRONÓSTICO EN EL BORDE
# ============================================================
# Grupo de control: los clientes que el estudio de caída marcó SIN_CAIDA.
# Su ventana reciente termina en el último mes COMPLETO (el estudio ya aplica
# el retroceso), así que sabemos que su consumo está estable. Si para esos
# clientes el pronóstico proyecta una baja, esa baja es del modelo, no del
# cliente — y si aparece solo en rurales, es el borde truncado.
# ============================================================

if HAY_PRONOSTICO:

    control = caida[caida["veredicto"] == "SIN_CAIDA"][
        ["NIU", "zona", "consumo_reciente_kwh", "meses_ventana"]
    ].merge(pronostico[["NIU", "pred_1m_kwh"]], on="NIU", how="inner")

    control["pred_ventana"] = control["pred_1m_kwh"]

    valido = (control["consumo_reciente_kwh"] > 0) & control["pred_ventana"].notna()
    control = control[valido].copy()
    control["desvio_pct"] = (
        (control["pred_ventana"] - control["consumo_reciente_kwh"])
        / control["consumo_reciente_kwh"] * 100
    )

    sesgo = (
        control.groupby("zona")["desvio_pct"]
        .agg(n="size", mediana="median", p25=lambda s: s.quantile(0.25),
             p75=lambda s: s.quantile(0.75))
        .round(2)
    )

    print("SESGO DEL PRONÓSTICO SOBRE CLIENTES ESTABLES")
    print("=" * 78)
    print("Estos clientes NO cayeron. Un pronóstico sin sesgo debería proyectarlos")
    print("cerca de 0%. Lo que se desvíe de ahí es del modelo, no del cliente.\n")
    display(sesgo)

    sesgo.reset_index().to_csv(RUTA_DIAGNOSTICO_SESGO, index=False, encoding="utf-8-sig")

    if {"RURAL", "URBANO"}.issubset(set(sesgo.index)):
        sesgo_rural = float(sesgo.loc["RURAL", "mediana"])
        sesgo_urbano = float(sesgo.loc["URBANO", "mediana"])
        brecha = sesgo_rural - sesgo_urbano

        print(f"\nRural  : {sesgo_rural:+.1f}%")
        print(f"Urbano : {sesgo_urbano:+.1f}%")
        print(f"Brecha : {brecha:+.1f} puntos porcentuales")

        # --- Decisión: ¿entran los rurales a la trayectoria? ---
        if EXCLUIR_RURAL_DE_TRAYECTORIA is None:
            excluir_rural = abs(brecha) > BRECHA_MAXIMA_ACEPTABLE_PP
            origen_decision = "automática, por la brecha medida"
        else:
            excluir_rural = bool(EXCLUIR_RURAL_DE_TRAYECTORIA)
            origen_decision = "forzada en la configuración"

        print(f"\nDECISIÓN ({origen_decision})")
        print("-" * 78)
        if excluir_rural:
            print(
                f"⚠ Brecha de {brecha:+.1f} pp supera el tope de "
                f"±{BRECHA_MAXIMA_ACEPTABLE_PP:.0f}.\n"
                "  El pronóstico rural arranca de una base distinta a la urbana "
                "(borde\n  provisional). Los rurales NO reciben trayectoria."
            )
        else:
            print(
                f"✓ Brecha de {brecha:+.1f} pp dentro del tope de "
                f"±{BRECHA_MAXIMA_ACEPTABLE_PP:.0f}.\n"
                "  El pronóstico trata igual a rurales y urbanos: los rurales "
                "SÍ reciben\n  trayectoria, con el WAPE a 1 mes de su perfil."
            )
    else:
        excluir_rural = (
            True if EXCLUIR_RURAL_DE_TRAYECTORIA is None
            else bool(EXCLUIR_RURAL_DE_TRAYECTORIA)
        )
        print("\nNo hay ambas zonas en el control: no se puede medir la brecha.")


In [ ]:
# ============================================================
# 6d. CLASIFICAR LA TRAYECTORIA
# ============================================================

if not HAY_PRONOSTICO:
    excluir_rural = True
    gestion["trayectoria"] = "SIN_PRONOSTICO"
    gestion["variacion_proyectada_pct"] = np.nan
    gestion["wape_aplicable_pct"] = np.nan
else:
    gestion = gestion.merge(
        pronostico[["NIU", "perfil", "fecha_pred_1m", "pred_1m_kwh"]],
        on="NIU", how="left",
    )

    # Solo el MES SIGUIENTE. Es el horizonte más preciso del modelo, el que
    # se gestiona en la práctica, y su mes de examen en el backtest queda
    # lejos del borde provisional. Los promedios a 3 y 6 meses acumulan
    # error sin aportar nada a una decisión que se toma mes a mes.
    gestion["horizonte_usado"] = 1
    gestion["pred_ventana_kwh"] = gestion["pred_1m_kwh"]

    gestion["variacion_proyectada_pct"] = np.where(
        gestion["consumo_reciente_kwh"] > 0,
        (gestion["pred_ventana_kwh"] - gestion["consumo_reciente_kwh"])
        / gestion["consumo_reciente_kwh"] * 100,
        np.nan,
    )

    # Umbral de credibilidad: el error del modelo PARA ESE perfil y horizonte
    idx = pd.MultiIndex.from_arrays(
        [gestion["perfil"], gestion["horizonte_usado"]]
    )
    gestion["wape_aplicable_pct"] = wape_por_grupo.reindex(idx).to_numpy()

    sin_pron = gestion["variacion_proyectada_pct"].isna() | gestion["wape_aplicable_pct"].isna()
    es_rural = gestion["zona"].eq("RURAL")
    supera = gestion["variacion_proyectada_pct"].abs() > gestion["wape_aplicable_pct"]
    baja = gestion["variacion_proyectada_pct"] < 0

    condiciones = [
        sin_pron,
        es_rural if excluir_rural else pd.Series(False, index=gestion.index),
        supera & baja,
        supera & ~baja,
    ]
    resultados = [
        "SIN_PRONOSTICO",
        "NO_EVALUABLE_RURAL",
        "CAIDA_ACELERANDO",
        "RECUPERACION_PREVISTA",
    ]
    gestion["trayectoria"] = np.select(
        condiciones, resultados, default="SIN_RECUPERACION_PREVISTA",
    )

print("TRAYECTORIA")
print("=" * 78)
display(gestion["trayectoria"].value_counts().rename("n_clientes").to_frame())

print("\nCÓMO LEER CADA VALOR")
print("-" * 78)
print("  (Todo se evalúa contra el pronóstico del MES SIGUIENTE, no promedios.)")
print("  CAIDA_ACELERANDO          : el modelo proyecta que el mes que viene sigue")
print("                              bajando desde el nivel ya caído. Máxima urgencia.")
print("  SIN_RECUPERACION_PREVISTA : el modelo no proyecta un cambio mayor a su")
print("                              propio error. No se ve recuperación — que no")
print("                              es lo mismo que garantizar que siga igual.")
print("  RECUPERACION_PREVISTA     : proyecta que vuelve. Vigilar, no despachar.")
print("  NO_EVALUABLE_RURAL        : el pronóstico arranca desde el mes truncado.")
print("                              Ver el sesgo medido en la celda anterior.")
print("  SIN_PRONOSTICO            : el cliente no tiene fila en el pronóstico.")

if HAY_PRONOSTICO:
    print("\nTRAYECTORIA POR ZONA")
    print("-" * 78)
    display(pd.crosstab(gestion["zona"], gestion["trayectoria"]))


## Historial: ¿desde cuándo está cada cliente en la lista?


In [ ]:
# ============================================================
# 6e. RECURRENCIA: CRUZAR CON LAS LISTAS DE LOS CORTES ANTERIORES
# ============================================================
# Un cliente que aparece por primera vez no es lo mismo que uno que lleva
# tres meses en la lista sin que nadie lo haya visitado. Esto se sabe solo
# si se conserva la lista de cada mes (carpeta historial/).
# ============================================================

patron_historial = re.compile(r"^gestion_caida_operativa_corte_(\d{4}-\d{2})\.csv$")

listas_previas = {}
for ruta in sorted(HISTORIAL_GESTION_DIR.glob("gestion_caida_operativa_corte_*.csv")):
    m = patron_historial.match(ruta.name)
    if not m:
        continue
    corte = pd.Timestamp(m.group(1) + "-01")
    if corte >= FECHA_CORTE:
        continue  # el corte actual (si se está recorriendo) o uno futuro no cuentan
    listas_previas[corte] = set(
        pd.read_csv(ruta, usecols=["NIU"], dtype={"NIU": "string"})["NIU"].str.strip()
    )

cortes_previos = sorted(listas_previas)

print("HISTORIAL DE LISTAS ANTERIORES")
print("-" * 70)
if not cortes_previos:
    print("No hay listas de cortes anteriores en", HISTORIAL_GESTION_DIR)
    print("Es la primera corrida con historial: todos los clientes cuentan como NUEVO.")
else:
    print(f"Cortes disponibles: {', '.join(f'{c:%Y-%m}' for c in cortes_previos)}")

def meses_entre(a, b):
    return (b.year - a.year) * 12 + (b.month - a.month)

# Consecutivos hacia atrás desde el corte actual: se detiene en el primer
# mes en que el cliente NO estaba (o en que no existe lista).
consecutivos = pd.Series(1, index=gestion["NIU"].astype("string"), dtype="int64")
for k in range(1, 25):
    corte_k = FECHA_CORTE - pd.DateOffset(months=k)
    if corte_k not in listas_previas:
        break
    en_lista_k = consecutivos.index.isin(listas_previas[corte_k])
    # solo siguen sumando los que llevan racha completa hasta aquí
    racha_viva = consecutivos.eq(k)
    consecutivos[racha_viva & en_lista_k] = k + 1

veces_12m = pd.Series(0, index=consecutivos.index, dtype="int64")
for corte, nius in listas_previas.items():
    if 1 <= meses_entre(corte, FECHA_CORTE) <= 12:
        veces_12m[veces_12m.index.isin(nius)] += 1

gestion["fecha_corte"] = FECHA_CORTE
gestion["meses_consecutivos_en_lista"] = consecutivos.to_numpy()
gestion["veces_en_lista_12m"] = veces_12m.to_numpy()
gestion["estado_en_lista"] = np.select(
    [
        gestion["meses_consecutivos_en_lista"] >= 2,
        gestion["veces_en_lista_12m"] >= 1,
    ],
    ["PERSISTENTE", "REINCIDENTE"],
    default="NUEVO",
)

print("\nESTADO EN LA LISTA")
print("-" * 70)
print("  NUEVO       : primera vez en la lista (o sin historial)")
print("  PERSISTENTE : también estaba el mes pasado (lleva N meses seguidos)")
print("  REINCIDENTE : estuvo en algún corte de los últimos 12 meses, pero no el pasado")
display(
    gestion.groupby("estado_en_lista")
    .agg(n_clientes=("NIU", "size"), kwh_riesgo_mes=("perdida_kwh_mes", "sum"))
    .assign(pct=lambda d: (d["n_clientes"] / len(gestion) * 100).round(1))
)
if gestion["meses_consecutivos_en_lista"].max() >= 2:
    display(
        gestion["meses_consecutivos_en_lista"].value_counts().sort_index()
        .rename("n_clientes").to_frame()
    )

# --- Entradas y salidas respecto al corte anterior ---
if cortes_previos:
    corte_previo = cortes_previos[-1]
    previos = listas_previas[corte_previo]
    actuales = set(gestion["NIU"].astype("string"))
    entradas_salidas = pd.DataFrame([{
        "fecha_corte": ETIQUETA_CORTE,
        "corte_anterior": f"{corte_previo:%Y-%m}",
        "en_lista_anterior": len(previos),
        "en_lista_actual": len(actuales),
        "permanecen": len(previos & actuales),
        "salieron": len(previos - actuales),
        "entraron": len(actuales - previos),
    }])
    print(f"\nRESPECTO AL CORTE ANTERIOR ({corte_previo:%Y-%m})")
    print("-" * 70)
    display(entradas_salidas.T)
    print("Un cliente que 'salió' dejó de cumplir el criterio de caída este mes: "
          "puede haberse recuperado, o haber caído tanto que pasó a SIN_CONSUMO.")
    # el resumen acumula una fila por corte
    if RUTA_ENTRADAS_SALIDAS.exists():
        anterior = pd.read_csv(RUTA_ENTRADAS_SALIDAS, dtype={"fecha_corte": "string"})
        anterior = anterior[anterior["fecha_corte"] != ETIQUETA_CORTE]
        entradas_salidas = pd.concat([anterior, entradas_salidas], ignore_index=True)
    entradas_salidas.to_csv(RUTA_ENTRADAS_SALIDAS, index=False, encoding="utf-8-sig")
    print("Guardado:", RUTA_ENTRADAS_SALIDAS)


## Vista operativa — por ciclo de lectura

In [ ]:
# ============================================================
# 7. LISTA OPERATIVA: ORDENADA POR CICLO
# ============================================================
# Dentro de cada ciclo, los de mayor valor primero. Así una cuadrilla que
# atiende un ciclo sabe por dónde empezar sin tener que ordenar nada.
# ============================================================

gestion["ciclo_orden"] = gestion["ciclo"].fillna(999)
gestion["ciclo_etiqueta"] = np.where(
    gestion["ciclo"].isna(),
    "SIN_CICLO",
    gestion["ciclo"].astype("Float64").astype("object").apply(
        lambda x: f"{int(x):02d}" if pd.notna(x) else "SIN_CICLO"
    ),
)

operativa = gestion.sort_values(
    ["ciclo_orden", COLUMNA_ORDEN],
    ascending=[True, False],
).copy()

operativa["orden_en_ciclo"] = (
    operativa.groupby("ciclo_etiqueta").cumcount() + 1
)

COLUMNAS_OPERATIVA = [
    "ciclo_etiqueta", "orden_en_ciclo", "NIU",
    "zona", "clase_servicio", "estrato", "tramo_consumo",
    "cluster_id", "veredicto", "severidad",
    "consumo_anterior_kwh", "consumo_reciente_kwh", "perdida_kwh_mes",
    "valor_riesgo_mes", "tarifa_kwh", "tiene_tarifa", "valor_orden",
    "variacion_vs_anterior_pct", "variacion_vs_anio_pct", "meses_ventana",
    "trayectoria", "pred_1m_kwh", "variacion_proyectada_pct", "wape_aplicable_pct",
    "estado_en_lista", "meses_consecutivos_en_lista", "veces_en_lista_12m", "fecha_corte",
]
COLUMNAS_OPERATIVA = [c for c in COLUMNAS_OPERATIVA if c in operativa.columns]

operativa[COLUMNAS_OPERATIVA].to_csv(
    RUTA_OPERATIVA, index=False, encoding="utf-8-sig",
)

# Copia por corte (no se sobreescribe el mes siguiente)
RUTA_OPERATIVA_HISTORIAL = (
    HISTORIAL_GESTION_DIR / f"gestion_caida_operativa_corte_{ETIQUETA_CORTE}.csv"
)
operativa[COLUMNAS_OPERATIVA].to_csv(
    RUTA_OPERATIVA_HISTORIAL, index=False, encoding="utf-8-sig",
)
print("Copia versionada:", RUTA_OPERATIVA_HISTORIAL)

print(f"Lista operativa: {len(operativa):,} clientes en "
      f"{operativa['ciclo_etiqueta'].nunique()} ciclos")
print("Guardado:", RUTA_OPERATIVA)

print("\nPRIMEROS 20 (ciclo más bajo primero, mayor valor dentro del ciclo)")
print("=" * 78)
display(operativa[COLUMNAS_OPERATIVA].head(20))


In [ ]:
# ============================================================
# 8. RESUMEN POR CICLO — CUÁNTO VALE CADA RUTA
# ============================================================

agregados = {
    "n_clientes": ("NIU", "size"),
    "kwh_riesgo_mes": ("perdida_kwh_mes", "sum"),
    "n_criticos": ("severidad", lambda s: int((s == "CRITICA").sum())),
    "n_rural": ("zona", lambda s: int((s == "RURAL").sum())),
}
if USAR_PESOS:
    agregados["valor_riesgo_mes"] = ("valor_riesgo_mes", "sum")

resumen_ciclo = (
    gestion.groupby("ciclo_etiqueta").agg(**agregados).reset_index()
)

columna_valor = "valor_riesgo_mes" if USAR_PESOS else "kwh_riesgo_mes"
resumen_ciclo = resumen_ciclo.sort_values(columna_valor, ascending=False)
resumen_ciclo["pct_del_total"] = (
    resumen_ciclo[columna_valor] / resumen_ciclo[columna_valor].sum() * 100
).round(2)
resumen_ciclo["acumulado_pct"] = resumen_ciclo["pct_del_total"].cumsum().round(2)

display(resumen_ciclo.round(0))

resumen_ciclo.to_csv(RUTA_RESUMEN_CICLO, index=False, encoding="utf-8-sig")
print("\nGuardado:", RUTA_RESUMEN_CICLO)

# ¿Cuántas rutas cubren la mayor parte del valor?
for objetivo in [50, 80]:
    n_rutas = int((resumen_ciclo["acumulado_pct"] <= objetivo).sum()) + 1
    n_rutas = min(n_rutas, len(resumen_ciclo))
    print(f"  {n_rutas} ciclos concentran el {objetivo}% del valor en riesgo")


## Vista gerencial y cortes de gestión

In [ ]:
# ============================================================
# 9. LISTA GERENCIAL: RANKING GLOBAL POR VALOR
# ============================================================

gerencial = gestion.sort_values(COLUMNA_ORDEN, ascending=False).copy()
gerencial["ranking"] = np.arange(1, len(gerencial) + 1)

COLUMNAS_GERENCIAL = ["ranking"] + [
    c for c in COLUMNAS_OPERATIVA if c not in ("orden_en_ciclo",)
]
COLUMNAS_GERENCIAL = [c for c in COLUMNAS_GERENCIAL if c in gerencial.columns]

gerencial[COLUMNAS_GERENCIAL].to_csv(
    RUTA_GERENCIAL, index=False, encoding="utf-8-sig",
)
print("Guardado:", RUTA_GERENCIAL)

print("\nTOP 25 POR VALOR EN RIESGO")
print("=" * 78)
display(gerencial[COLUMNAS_GERENCIAL].head(25))

# Los clientes sin tarifa no tienen valor, así que caen al fondo del ranking.
# Se reportan aparte para que no desaparezcan de la vista.
if USAR_PESOS:
    sin_tarifa = gestion[~gestion["tiene_tarifa"]].sort_values(
        "perdida_kwh_mes", ascending=False
    )
    if len(sin_tarifa):
        print(f"\nCLIENTES SIN TARIFA — {len(sin_tarifa):,}, sin valorar")
        print("-" * 78)
        print(f"Suman {sin_tarifa['perdida_kwh_mes'].sum():,.0f} kWh/mes. "
              "Ordenados por kWh, van los 10 mayores:")
        display(
            sin_tarifa[[c for c in COLUMNAS_OPERATIVA if c in sin_tarifa.columns]].head(10)
        )
        print("Resolver su tarifa en el sistema comercial los incorpora al ranking.")

print("\nCONCENTRACIÓN")
print("-" * 78)
columna_concentracion = "valor_riesgo_mes" if USAR_PESOS else "perdida_kwh_mes"
total = gerencial[columna_concentracion].sum()
for n in [10, 50, 100, 500, 1000]:
    if n <= len(gerencial):
        parte = gerencial[columna_concentracion].head(n).sum()
        unidad = "$" if USAR_PESOS else "kWh"
        print(f"  Top {n:>5}: {unidad}{parte:>15,.0f}/mes  "
              f"({parte / total * 100:5.1f}% del total)")


In [ ]:
# ============================================================
# 10. RESUMEN POR CORTES DE GESTIÓN
# ============================================================
# Clase de servicio, estrato, zona y tramo: cada uno reparte a un área
# distinta de la empresa.
# ============================================================

def resumir_por(columna):
    agg = {
        "n_clientes": ("NIU", "size"),
        "kwh_riesgo_mes": ("perdida_kwh_mes", "sum"),
        "n_criticos": ("severidad", lambda s: int((s == "CRITICA").sum())),
    }
    if USAR_PESOS:
        agg["valor_riesgo_mes"] = ("valor_riesgo_mes", "sum")

    salida = (
        gestion.groupby(columna, dropna=False)
        .agg(**agg)
        .reset_index()
        .rename(columns={columna: "valor_corte"})
    )
    salida.insert(0, "corte", columna)
    salida["valor_corte"] = salida["valor_corte"].astype("string")
    return salida.sort_values(columna_valor, ascending=False)


cortes = pd.concat(
    [resumir_por(c) for c in
     ["clase_servicio", "estrato", "zona", "tramo_consumo"]
     if c in gestion.columns],
    ignore_index=True,
)

for corte, grupo in cortes.groupby("corte", sort=False):
    print(f"\n{corte.upper()}")
    print("-" * 78)
    display(grupo.drop(columns="corte").round(0))

cortes.to_csv(RUTA_RESUMEN_CORTE, index=False, encoding="utf-8-sig")
print("\nGuardado:", RUTA_RESUMEN_CORTE)


In [ ]:
# ============================================================
# 11. GRÁFICAS
# ============================================================

unidad = "Valor en riesgo ($/mes)" if USAR_PESOS else "kWh en riesgo al mes"

# --- 1. Los 20 ciclos que más valor concentran ---
top_ciclos = resumen_ciclo.head(20)

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(top_ciclos["ciclo_etiqueta"], top_ciclos[columna_valor])
ax.set_xlabel("Ciclo de lectura")
ax.set_ylabel(unidad)
ax.set_title("Los 20 ciclos con más valor en riesgo — por dónde empezar")
ax.grid(alpha=0.25, axis="y")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- 2. Curva de concentración ---
fig, ax = plt.subplots(figsize=(9, 5.5))
acumulado = (
    gerencial[columna_concentracion].cumsum()
    / gerencial[columna_concentracion].sum() * 100
)
ax.plot(np.arange(1, len(acumulado) + 1), acumulado.to_numpy())
ax.axhline(80, color="red", linestyle="--", linewidth=0.9, label="80% del valor")
ax.set_xscale("log")
ax.set_xlabel("Número de clientes gestionados (escala log)")
ax.set_ylabel("% del valor en riesgo cubierto")
ax.set_title("Cuántos clientes hay que gestionar para cubrir el valor")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# --- 3. Composición por clase de servicio ---
comp = (
    gestion.groupby("clase_servicio")[columna_concentracion].sum()
    .sort_values(ascending=True)
)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(comp.index.astype(str), comp.to_numpy())
ax.set_xlabel(unidad)
ax.set_title("Valor en riesgo por clase de servicio")
ax.grid(alpha=0.25, axis="x")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 12. CIERRE
# ============================================================

print("PRIORIZACIÓN DE GESTIÓN — TERMINADA")
print("=" * 78)

print(f"\nClientes en la lista : {len(gestion):,}")
print(f"Veredictos incluidos : {', '.join(VEREDICTOS_GESTION)}")
print(f"Ciclos involucrados  : {gestion['ciclo_etiqueta'].nunique()}")
print(f"kWh en riesgo        : {gestion['perdida_kwh_mes'].sum():,.0f} /mes")
if USAR_PESOS:
    n_val = int(gestion["tiene_tarifa"].sum())
    print(f"Valor en riesgo      : ${gestion['valor_riesgo_mes'].sum():,.0f} /mes "
          f"(sobre {n_val:,} clientes con tarifa real)")
    n_sv = len(gestion) - n_val
    if n_sv:
        print(f"Sin valorar          : {n_sv:,} clientes, "
              f"{gestion.loc[~gestion['tiene_tarifa'], 'perdida_kwh_mes'].sum():,.0f} kWh/mes")
else:
    print("Valor en riesgo      : no calculado (la tarifa no pasó la auditoría)")

print("\nSalidas:")
print(" •", RUTA_OPERATIVA, "(por ciclo — para repartir a cuadrillas)")
print(" •", RUTA_GERENCIAL, "(ranking global por valor)")
print(" •", RUTA_RESUMEN_CICLO, "(cuánto vale cada ruta)")
print(" •", RUTA_RESUMEN_CORTE, "(clase, estrato, zona, tramo)")
print(" •", RUTA_AUDITORIA_TARIFA, "(auditoría del campo de tarifa)")
print(" •", RUTA_OPERATIVA_HISTORIAL, "(copia de este corte, no se sobreescribe)")
if RUTA_ENTRADAS_SALIDAS.exists():
    print(" •", RUTA_ENTRADAS_SALIDAS, "(entradas y salidas de la lista por corte)")
print(f"\nEstado en la lista: "
      + ", ".join(f"{k} {v:,}" for k, v in gestion["estado_en_lista"].value_counts().items()))
